# v06 - DICOM audit, slot selection, targets and folds (CPU)

First step of the own-model leg (`docs/research/v06-own-model-design.md`, revision 2,
decision D-005). Read-only with respect to the competition data: nothing is cached except
small tables. Runs on CPU; no GPU quota.

What it does:

1. **Header pass** over every training series: slice count, header errors, transfer
   syntax, geometry (`ImagePositionPatient`, `ImageOrientationPatient`), obliquity, pixel
   spacing, field of view, and scanner fingerprint fields.
2. **Slot selection** with the rule that v07 (cache) and v09 (test inference) reuse:
   5 slots, 32 slices per study.
3. **Decode check** of every selected slice (pixels are decoded and discarded).
4. **Pass/fail gates** from design section 4.1.
5. **Soft targets** (mean of four public report-label tables, UNK cells down-weighted)
   and **folds** (scanner-grouped if the fingerprint rule in design section 4.5 holds,
   otherwise random), both over the non-gold studies.

Inputs: the competition data and four public label datasets:
`pilkwang/rsna-knee-llm-labels`, `stevenleehans/rsna-knee-llm-report-labels`,
`lixin73/rsna-knee-llm-report-labels-sol56`, `dreaddevelopment/rsna-knee-labels`.

Outputs in `/kaggle/working/v06/`: `v06_series_audit.csv`, `v06_slot_selection.csv`,
`v06_study_audit.csv`, `v06_targets.csv`, `v06_folds.csv`, `v06_audit_summary.json`.

In [ ]:
# v06 configuration
import json
import os
import time
from pathlib import Path

VERSION = 'v06'
WORK = Path('/kaggle/working') / VERSION
WORK.mkdir(parents=True, exist_ok=True)
T0 = time.time()

# Set to a small integer for a quick dry run; None audits every training study.
DRY_RUN_STUDIES = None
N_WORKERS = max(1, os.cpu_count() or 1)
SEED = 2026

# (slot name, plane, fluid-sensitive preference, slices); see design section 4.2.
# Preference True = fluid-sensitive first, False = non-fluid-sensitive first,
# None = no preference (any remaining series of the plane).
SLOT_SPEC = [
    ('SAG_FS', 'Sagittal', True, 8),
    ('SAG_NFS', 'Sagittal', False, 6),
    ('COR_FS', 'Coronal', True, 8),
    ('COR_OTHER', 'Coronal', None, 4),
    ('AX_FS', 'Axial', True, 6),
]
CENTRAL_BAND = (0.10, 0.90)
CROP_MM = 140.0

# Gates (design section 4.1).
GATE_MIN_STUDY_DECODE_OK = 0.995
GATE_MIN_SERIES_GEOMETRY_OK = 0.99

# Labels and folds (design sections 4.3 and 4.5).
LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA',
          'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
LABEL_FILES = {
    'pilkwang_v2': 'report_labels_v2.csv',
    'steven_v2': 'llm_labels_v2.csv',
    'lixin_gpt56': 'labels_llm_gpt56sol.csv',
    'dread_soft': 'labels_llm_soft.csv',
}
UNK_WEIGHT = 0.3
N_FOLDS = 5
FP_MIN_COVERAGE = 0.90
FP_MIN_GROUPS = 10
FP_MAX_GROUP_SHARE = 0.40
FP_RARE_SHARE = 0.01
print(f'{VERSION}: workers={N_WORKERS}, dry_run={DRY_RUN_STUDIES}', flush=True)

In [ ]:
# Locate inputs (competition mount differs between Kaggle images).
import pandas as pd

def find_competition_root():
    for p in (Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'),
              Path('/kaggle/input/rsna-knee-abnormality-detection')):
        if (p / 'train_series.csv').is_file():
            return p
    raise FileNotFoundError('competition data not attached')

def find_unique_file(name):
    hits = sorted(Path('/kaggle/input').rglob(name))
    if len(hits) != 1:
        raise FileNotFoundError(f'expected one {name} under /kaggle/input, found {hits}')
    return hits[0]

COMP = find_competition_root()
IMAGE_ROOT = COMP / 'train_series'
assert IMAGE_ROOT.is_dir(), IMAGE_ROOT
train = pd.read_csv(COMP / 'train.csv', dtype={'StudyInstanceUID': str})
series_csv = pd.read_csv(COMP / 'train_series.csv',
                         dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
assert list(pd.read_csv(COMP / 'sample_submission.csv', nrows=0).columns[1:]) == LABELS
label_paths = {k: find_unique_file(v) for k, v in LABEL_FILES.items()}

studies = sorted(train.StudyInstanceUID.unique())
if DRY_RUN_STUDIES:
    studies = studies[:DRY_RUN_STUDIES]
series_csv = series_csv[series_csv.StudyInstanceUID.isin(studies)].reset_index(drop=True)
print(f'competition root: {COMP}')
print(f'studies: {len(studies)}, series: {len(series_csv)}')
print({k: str(v) for k, v in label_paths.items()})

## 1. Header pass

One worker call per series reads every file header (no pixels). Slices are ordered by the
projection of `ImagePositionPatient` on the slice normal; if geometry is missing or
degenerate, by `InstanceNumber`; otherwise by file name (flagged).

In [ ]:
import numpy as np
import pydicom
from concurrent.futures import ProcessPoolExecutor

HEADER_TAGS = ['ImagePositionPatient', 'ImageOrientationPatient', 'InstanceNumber',
               'PixelSpacing', 'Rows', 'Columns', 'NumberOfFrames', 'Manufacturer',
               'ManufacturerModelName', 'MagneticFieldStrength', 'Laterality',
               'SeriesDescription', 'StationName', 'InstitutionName']

def _floats(value, n):
    try:
        out = [float(v) for v in value]
    except Exception:
        return None
    return out if len(out) == n and all(np.isfinite(out)) else None

def _text(ds, key):
    value = getattr(ds, key, None)
    if value is None:
        return ''
    return str(value).strip()

def order_slices(records):
    # records: list of dicts with name, ipp, iop, inst. Returns (ordered names, method, info).
    info = {}
    with_geom = [r for r in records if r['ipp'] is not None and r['iop'] is not None]
    if records and len(with_geom) == len(records):
        iop = np.asarray(with_geom[0]['iop'], dtype=np.float64)
        normal = np.cross(iop[:3], iop[3:])
        norm = np.linalg.norm(normal)
        if norm > 1e-6:
            normal = normal / norm
            pos = np.asarray([np.dot(r['ipp'], normal) for r in with_geom])
            info['n_unique_positions'] = int(len(np.unique(np.round(pos, 3))))
            info['obliquity_deg'] = float(np.degrees(np.arccos(min(1.0, np.abs(normal).max()))))
            iop_all = np.asarray([r['iop'] for r in with_geom], dtype=np.float64)
            info['iop_consistent'] = bool(np.allclose(iop_all, iop_all[0], atol=1e-3))
            if info['n_unique_positions'] >= 0.9 * len(records):
                order = np.argsort(pos, kind='stable')
                gaps = np.diff(pos[order])
                info['slice_gap_mm'] = float(np.median(np.abs(gaps))) if len(gaps) else float('nan')
                return [with_geom[i]['name'] for i in order], 'geometry', info
    if records and all(r['inst'] is not None for r in records):
        ordered = sorted(records, key=lambda r: (r['inst'], r['name']))
        return [r['name'] for r in ordered], 'instance_number', info
    return sorted(r['name'] for r in records), 'filename', info

def audit_series(args):
    study, series = args
    sdir = IMAGE_ROOT / study / series
    row = {'StudyInstanceUID': study, 'SeriesInstanceUID': series, 'dir_exists': sdir.is_dir()}
    if not row['dir_exists']:
        return row, []
    names = sorted(p.name for p in sdir.iterdir() if p.suffix.lower() == '.dcm')
    row['n_files'] = len(names)
    records, errors, first = [], 0, None
    for name in names:
        try:
            ds = pydicom.dcmread(str(sdir / name), stop_before_pixels=True, force=True,
                                 specific_tags=HEADER_TAGS)
        except Exception:
            errors += 1
            continue
        if first is None:
            first = ds
            meta = getattr(ds, 'file_meta', None)
            row['transfer_syntax'] = str(getattr(meta, 'TransferSyntaxUID', '')) if meta else ''
        try:
            inst = int(float(str(getattr(ds, 'InstanceNumber', '')).strip()))
        except ValueError:
            inst = None
        records.append({'name': name,
                        'ipp': _floats(getattr(ds, 'ImagePositionPatient', None) or [], 3),
                        'iop': _floats(getattr(ds, 'ImageOrientationPatient', None) or [], 6),
                        'inst': inst})
    row['n_header_errors'] = errors
    row['n_readable'] = len(records)
    if first is not None:
        spacing = _floats(getattr(first, 'PixelSpacing', None) or [], 2)
        rows_, cols_ = getattr(first, 'Rows', None), getattr(first, 'Columns', None)
        row['pixel_spacing'] = spacing[0] if spacing else np.nan
        row['rows'], row['cols'] = rows_, cols_
        if spacing and rows_ and cols_:
            row['fov_mm'] = float(min(spacing[0] * int(rows_), spacing[1] * int(cols_)))
        row['n_frames'] = int(getattr(first, 'NumberOfFrames', 1) or 1)
        for key in ('Manufacturer', 'ManufacturerModelName', 'MagneticFieldStrength',
                    'Laterality', 'SeriesDescription', 'StationName', 'InstitutionName'):
            row[key] = _text(first, key)
    ordered, method, info = order_slices(records)
    row['sort_method'] = method
    row.update(info)
    return row, ordered

t = time.time()
jobs = list(zip(series_csv.StudyInstanceUID, series_csv.SeriesInstanceUID))
series_rows, ORDERED = [], {}
with ProcessPoolExecutor(N_WORKERS) as pool:
    for i, (row, ordered) in enumerate(pool.map(audit_series, jobs, chunksize=8)):
        series_rows.append(row)
        ORDERED[row['SeriesInstanceUID']] = ordered
        if (i + 1) % 2000 == 0:
            print(f'  headers: {i + 1}/{len(jobs)} series, {time.time() - t:.0f}s', flush=True)
series_audit = series_csv.merge(pd.DataFrame(series_rows), on=['StudyInstanceUID', 'SeriesInstanceUID'], how='left')
series_audit['geometry_ok'] = (series_audit.sort_method == 'geometry') & series_audit.pixel_spacing.notna()
print(f'header pass: {len(series_audit)} series in {time.time() - t:.0f}s', flush=True)
print(series_audit.sort_method.value_counts(dropna=False).to_string())

## 2. Slot selection (shared rule)

Candidates are ranked by: usable geometry first, then more readable slices, then series
UID for determinism. Paired slots of one plane take different series when the study has
them; a duplicate is used only when the plane has a single series, and is flagged.

In [ ]:
def slice_positions(n, k, band=CENTRAL_BAND):
    # k indices evenly spaced over the central band of an ordered series of length n.
    if n <= 0:
        return []
    lo, hi = band[0] * (n - 1), band[1] * (n - 1)
    return [int(round(x)) for x in np.linspace(lo, hi, k)]

def _rank_key(c):
    return (0 if c['geometry_ok'] else 1, -int(c['n_readable'] or 0), c['SeriesInstanceUID'])

def select_slots(study_series):
    # study_series: list of dicts with SeriesInstanceUID, Anatomical_Plane, Fluid_Sensitive,
    # geometry_ok, n_readable. Returns {slot: (series uid or None, duplicate flag, fallback flag)}.
    usable = [s for s in study_series if (s.get('n_readable') or 0) > 0]
    chosen, used = {}, set()
    for slot, plane, fluid_pref, _k in SLOT_SPEC:
        same_plane = sorted((s for s in usable if s['Anatomical_Plane'] == plane), key=_rank_key)
        if not same_plane:
            chosen[slot] = (None, False, False)
            continue
        unused = [s for s in same_plane if s['SeriesInstanceUID'] not in used]
        if fluid_pref is None:
            preferred = sorted(unused, key=lambda s: (int(s['Fluid_Sensitive']),) + _rank_key(s))
        else:
            preferred = [s for s in unused if bool(s['Fluid_Sensitive']) == fluid_pref]
        if preferred:
            pick, duplicate, fallback = preferred[0], False, False
        elif unused:
            pick, duplicate, fallback = unused[0], False, True
        else:
            pick, duplicate, fallback = same_plane[0], True, True
        used.add(pick['SeriesInstanceUID'])
        chosen[slot] = (pick['SeriesInstanceUID'], duplicate, fallback)
    return chosen

selection_rows = []
by_study = {k: g.to_dict('records') for k, g in series_audit.groupby('StudyInstanceUID')}
for study in studies:
    chosen = select_slots(by_study.get(study, []))
    for slot, plane, _pref, k in SLOT_SPEC:
        uid, duplicate, fallback = chosen[slot]
        files = ORDERED.get(uid, []) if uid else []
        idx = slice_positions(len(files), k)
        selection_rows.append({'StudyInstanceUID': study, 'slot': slot, 'plane': plane,
                               'SeriesInstanceUID': uid, 'duplicate': duplicate,
                               'fallback': fallback, 'n_series_slices': len(files),
                               'slice_files': json.dumps([files[i] for i in idx])})
slot_selection = pd.DataFrame(selection_rows)
print(slot_selection.groupby('slot')[['duplicate', 'fallback']].mean().round(4).to_string())
print('missing slots:', int(slot_selection.SeriesInstanceUID.isna().sum()))

## 3. Decode check

Every selected slice is decoded with pydicom and checked for a finite, non-constant 2D
array. Pixels are discarded.

In [ ]:
def pixel_backends():
    out = {'pydicom': pydicom.__version__}
    for mod in ('pylibjpeg', 'openjpeg', 'libjpeg', 'gdcm'):
        try:
            out[mod] = getattr(__import__(mod), '__version__', 'present')
        except Exception:
            out[mod] = None
    return out

def decode_study(args):
    study, items = args
    result = {'StudyInstanceUID': study, 'n_slices': 0, 'n_decode_errors': 0, 'error_example': ''}
    for uid, names in items:
        for name in names:
            result['n_slices'] += 1
            try:
                ds = pydicom.dcmread(str(IMAGE_ROOT / study / uid / name), force=True)
                arr = ds.pixel_array
                if arr.ndim != 2 or not np.isfinite(arr).all() or arr.max() == arr.min():
                    raise ValueError(f'bad array shape={arr.shape} min={arr.min()} max={arr.max()}')
            except Exception as exc:
                result['n_decode_errors'] += 1
                if not result['error_example']:
                    result['error_example'] = f'{uid}/{name}: {type(exc).__name__}: {str(exc)[:200]}'
    return result

print('pixel backends:', pixel_backends(), flush=True)
t = time.time()
decode_jobs = []
for study, g in slot_selection.dropna(subset=['SeriesInstanceUID']).groupby('StudyInstanceUID'):
    decode_jobs.append((study, [(r.SeriesInstanceUID, json.loads(r.slice_files)) for r in g.itertuples()]))
decode_rows = []
with ProcessPoolExecutor(N_WORKERS) as pool:
    for i, row in enumerate(pool.map(decode_study, decode_jobs, chunksize=4)):
        decode_rows.append(row)
        if (i + 1) % 500 == 0:
            print(f'  decode: {i + 1}/{len(decode_jobs)} studies, {time.time() - t:.0f}s', flush=True)
decode_audit = pd.DataFrame(decode_rows)
print(f'decode check: {decode_audit.n_slices.sum()} slices in {time.time() - t:.0f}s, '
      f'errors={decode_audit.n_decode_errors.sum()}', flush=True)

## 4. Gates (design section 4.1)

In [ ]:
sel = slot_selection.dropna(subset=['SeriesInstanceUID'])
chosen_series = series_audit.set_index('SeriesInstanceUID').loc[sel.SeriesInstanceUID.unique()]
study_audit = (slot_selection.assign(present=slot_selection.SeriesInstanceUID.notna())
               .pivot_table(index='StudyInstanceUID', columns='slot', values='present', aggfunc='first')
               .add_prefix('has_').reset_index())
study_audit = study_audit.merge(decode_audit, on='StudyInstanceUID', how='left')
study_audit['n_decode_errors'] = study_audit.n_decode_errors.fillna(0).astype(int)
study_audit['decode_ok'] = study_audit.n_decode_errors == 0
study_audit['planes_present'] = study_audit[['has_SAG_FS', 'has_COR_FS', 'has_AX_FS']].fillna(False).astype(bool).sum(axis=1)

# A duplicate is avoidable only if the plane had another usable series; select_slots never
# does that by construction, so this check guards the rule itself.
plane_counts = (series_audit[series_audit.n_readable.fillna(0) > 0]
                .groupby(['StudyInstanceUID', 'Anatomical_Plane']).size())
dup = slot_selection[slot_selection.duplicate]
avoidable = sum(plane_counts.get((r.StudyInstanceUID, r.plane), 0) >= 2 for r in dup.itertuples())

gates = {
    'study_decode_ok_fraction': float(study_audit.decode_ok.mean()),
    'chosen_series_geometry_ok_fraction': float(chosen_series.geometry_ok.mean()),
    'avoidable_duplicate_slots': int(avoidable),
}
gates['pass_decode'] = gates['study_decode_ok_fraction'] >= GATE_MIN_STUDY_DECODE_OK
gates['pass_geometry'] = gates['chosen_series_geometry_ok_fraction'] >= GATE_MIN_SERIES_GEOMETRY_OK
gates['pass_no_avoidable_duplicates'] = gates['avoidable_duplicate_slots'] == 0
gates['pass_all'] = gates['pass_decode'] and gates['pass_geometry'] and gates['pass_no_avoidable_duplicates']
print(json.dumps(gates, indent=2))
print('chosen series FOV below crop:', float((chosen_series.fov_mm < CROP_MM).mean()))
print('chosen series obliquity > 10 deg:', float((chosen_series.obliquity_deg > 10).mean()))
print('transfer syntaxes:', chosen_series.transfer_syntax.value_counts().to_dict())
print(study_audit[~study_audit.decode_ok][['StudyInstanceUID', 'n_decode_errors', 'error_example']].head(10).to_string())

## 5. Soft targets (design section 4.3)

Mean probability over the four independent tables (missing cells averaged over available
sources). Loss weight `UNK_WEIGHT` where pilkwang's verdict is UNK (report silent), else 1.
The 58 gold studies keep their gold labels for evaluation and are excluded from folds.

In [ ]:
tables = {}
for key, path in label_paths.items():
    df = pd.read_csv(path, dtype={'StudyInstanceUID': str}).drop_duplicates('StudyInstanceUID')
    tables[key] = df.set_index('StudyInstanceUID')
stack = np.stack([tables[k].reindex(studies)[LABELS].to_numpy(np.float64) for k in tables])
soft = np.nanmean(stack, axis=0)
n_sources = np.isfinite(stack).sum(axis=0)
verdict = tables['pilkwang_v2'].reindex(studies)[[f'{c}__verdict' for c in LABELS]].to_numpy()
weight = np.where(verdict == 'UNK', UNK_WEIGHT, 1.0)

gold = train.set_index('StudyInstanceUID').reindex(studies)[LABELS]
is_gold = gold.notna().all(axis=1).to_numpy()
targets = pd.DataFrame(soft, columns=LABELS, index=studies)
targets[[f'{c}__weight' for c in LABELS]] = weight
targets[[f'{c}__n_sources' for c in LABELS]] = n_sources
targets['is_gold'] = is_gold
for c in LABELS:
    targets[f'{c}__gold'] = gold[c].to_numpy()
targets.index.name = 'StudyInstanceUID'
nongold_missing = np.isnan(soft[~is_gold]).any(axis=1).sum()
print('studies:', len(targets), 'gold:', int(is_gold.sum()),
      'non-gold rows with a missing target:', int(nongold_missing))
print('min sources per cell (non-gold):', int(n_sources[~is_gold].min()))
print('UNK share per label:', dict(zip(LABELS, np.round((verdict == 'UNK').mean(0), 3))))

## 6. Scanner fingerprints and folds (design section 4.5)

Fingerprint = manufacturer | model | field strength (tesla) of the chosen series, taken as
the most common value across a study's chosen series. Folds are grouped by fingerprint if
coverage, group count and largest-group share pass; otherwise each study is its own group
(random folds) and rare-fingerprint studies are flagged as the cross-site proxy.
Assignment is a greedy multilabel balance over groups.

In [ ]:
def field_tesla(text):
    try:
        v = float(text)
    except Exception:
        return None
    if v > 100:  # gauss
        v = v / 10000.0
    return round(v, 1) if v > 0 else None

def series_fingerprint(r):
    maker = (r.get('Manufacturer') or '').upper().split(' ')[0]
    model = (r.get('ManufacturerModelName') or '').upper()
    tesla = field_tesla(r.get('MagneticFieldStrength'))
    if not maker or not model or tesla is None:
        return None
    return f'{maker}|{model}|{tesla}'

series_audit['fingerprint'] = [series_fingerprint(r) for r in series_audit.to_dict('records')]
fp_by_series = series_audit.set_index('SeriesInstanceUID').fingerprint
study_fp = {}
for study, g in sel.groupby('StudyInstanceUID'):
    values = [fp_by_series.get(u) for u in g.SeriesInstanceUID if fp_by_series.get(u)]
    study_fp[study] = pd.Series(values).mode().iloc[0] if values else None
targets['fingerprint'] = [study_fp.get(s) for s in targets.index]

pool_ids = targets.index[~targets.is_gold]
fp = targets.loc[pool_ids, 'fingerprint']
coverage = float(fp.notna().mean())
shares = fp.value_counts(normalize=True)
fp_stats = {'coverage': coverage, 'n_groups': int(shares.size),
            'max_group_share': float(shares.max()) if shares.size else 1.0}
grouped = (coverage >= FP_MIN_COVERAGE and fp_stats['n_groups'] >= FP_MIN_GROUPS
           and fp_stats['max_group_share'] <= FP_MAX_GROUP_SHARE)
fp_stats['fold_scheme'] = 'scanner_grouped' if grouped else 'random'
print(json.dumps(fp_stats, indent=2))
print(shares.head(15).round(4).to_string())

def greedy_group_folds(group_ids, Y, n_folds, seed):
    # Assign whole groups to folds, balancing study count and per-label positives.
    rng = np.random.default_rng(seed)
    frame = pd.DataFrame(Y, index=group_ids.index)
    frame['_n'] = 1
    per_group = frame.groupby(group_ids.values).sum()
    order = per_group.sample(frac=1.0, random_state=int(rng.integers(1 << 31)))
    order = order.iloc[np.argsort(-order['_n'].to_numpy(), kind='stable')]
    target = per_group.sum().to_numpy(np.float64) / n_folds + 1e-9
    load = np.zeros((n_folds, per_group.shape[1]))
    # The study count weighs as much as all labels together, so fold sizes stay even.
    col_weight = np.ones(per_group.shape[1])
    col_weight[-1] = per_group.shape[1] - 1
    assign = {}
    for g, v in zip(order.index, order.to_numpy(np.float64)):
        cost = ((((load + v) / target) ** 2) * col_weight).sum(axis=1)
        f = int(np.argmin(cost))
        load[f] += v
        assign[g] = f
    return group_ids.map(assign).astype(int)

Y = (targets.loc[pool_ids, LABELS].to_numpy() >= 0.5).astype(int)
if grouped:
    groups = fp.fillna('unknown:' + fp.index.to_series())
else:
    groups = pd.Series(pool_ids, index=pool_ids)
fold = greedy_group_folds(groups, Y, N_FOLDS, SEED)
folds = pd.DataFrame({'StudyInstanceUID': targets.index, 'fold': -1})
folds = folds.set_index('StudyInstanceUID')
folds.loc[pool_ids, 'fold'] = fold.to_numpy()
folds['fold_scheme'] = fp_stats['fold_scheme']
folds['fingerprint'] = targets.fingerprint
rare = set(shares[shares < FP_RARE_SHARE].index)
folds['rare_fingerprint'] = [f in rare or f is None for f in folds.fingerprint]
balance = (pd.DataFrame(Y, index=pool_ids, columns=LABELS).assign(fold=fold.to_numpy())
           .groupby('fold').agg(['mean']).round(3))
print(folds[folds.fold >= 0].fold.value_counts().sort_index().to_string())
print(balance.to_string())

## 7. Write outputs

In [ ]:
series_out = series_audit.drop(columns=['StationName', 'InstitutionName'], errors='ignore')
series_out.to_csv(WORK / 'v06_series_audit.csv', index=False)
slot_selection.to_csv(WORK / 'v06_slot_selection.csv', index=False)
study_audit.to_csv(WORK / 'v06_study_audit.csv', index=False)
targets.to_csv(WORK / 'v06_targets.csv')
folds.to_csv(WORK / 'v06_folds.csv')
summary = {
    'version': VERSION,
    'dry_run_studies': DRY_RUN_STUDIES,
    'studies': len(studies),
    'series': int(len(series_audit)),
    'gates': gates,
    'fingerprints': fp_stats,
    'station_name_coverage': float((series_audit.StationName.fillna('') != '').mean()),
    'institution_name_coverage': float((series_audit.InstitutionName.fillna('') != '').mean()),
    'laterality_tag_coverage': float((series_audit.Laterality.fillna('') != '').mean()),
    'pixel_backends': pixel_backends(),
    'elapsed_seconds': round(time.time() - T0, 1),
}
(WORK / 'v06_audit_summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))